# Auditoría de producción de CO2: nutrición, oxígeno y activación

Tres respuestas nutricionales × dos opciones de oxígeno = **seis variantes**. Etapa 1: parámetros comunes congelados, transitorio apagado. Etapa 2: reajuste comparable de los mismos seis parámetros de la capa de CO2, incluido el transitorio. `theta_natural_full` permanece congelado.

Calibración: LAB005/LAB007/LAB008 (peso 1), LAB019/LAB020/LAB021 (peso 2). Validación: LAB011/LAB012. Holdout externo: LAB016/LAB017/LAB018. LAB004/LAB006 quedan fuera; LAB010 es sensibilidad, sin ajuste.

**Para explorar curvas:** ejecutar todo una vez, cambiar **`MANUAL` en la celda «Parámetros editables»**, volver a ejecutar esa celda y la última celda **«Explorador manual»**. No requiere cargar datos ni reajustar. `MANUAL` sólo afecta al explorador. Cambios en `AUDIT_CONFIG` requieren ejecutar desde esa celda hacia abajo.

Las figuras muestran curvas completas en los outputs. No se escriben CSV ni imágenes en `results`. SOURCE y EXECUTED son archivos separados; el notebook anterior permanece protegido.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import re
import sys
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython import get_ipython
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")
import numpy as np
import openpyxl
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "fermentation_model").exists())
FM = ROOT / "fermentation_model"
LAB2026 = FM / "laboratory_2026"
DATA_DIR = FM / "data" / "mem2026" / "LAB019-021"
DATA_016_018_DIR = FM / "data" / "mem2026" / "LAB016-018"
REFERENCE_RESULTS_DIR = LAB2026 / "results" / "lab019_021_theta_full_holdout"

for path in (FM, LAB2026, FM / "pilot_2025"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from shared import run_new_must_glycerol_estimability_doe as base
from shared import run_secondary_joint_campaign_doe as joint
from shared import run_secondary_v2_model_evaluation as secondary_v2
from laboratory_2026 import run_estimability_historical_by_medium as natural_loader
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2_cross

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9.5})

LABS = ("LAB019", "LAB020", "LAB021")
FERMENTER = {"LAB019": "F1", "LAB020": "F2", "LAB021": "F3"}
MATCHED_ORIGINAL = {"LAB019": "LAB004", "LAB020": "LAB006", "LAB021": "LAB009"}
COLORS = {"LAB019": "#4c72b0", "LAB020": "#55a868", "LAB021": "#c44e52"}
T0 = pd.Timestamp("2026-09-21 17:00:00")
# Hora y dosis reales del segundo pulso: hoja Nutricion, no un valor heredado del ajuste histórico.
nutrition_wb = openpyxl.load_workbook(
    DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx", read_only=True, data_only=True
)
pulse2_rows = [
    row for row in nutrition_wb["Nutricion"].iter_rows(min_row=2, values_only=True)
    if row[0] in LABS and row[2] == "Pulso 2"
]
nutrition_wb.close()
assert len(pulse2_rows) == len(LABS)
assert {row[0] for row in pulse2_rows} == set(LABS)
PULSE2_DT = {row[0]: pd.Timestamp(row[3]) for row in pulse2_rows}
PULSE2_H = {lab: (stamp - T0).total_seconds() / 3600.0 for lab, stamp in PULSE2_DT.items()}
pulse2_doses = {row[0]: float(row[4]) / 1000.0 for row in pulse2_rows}  # mg/L -> kg/m³
assert len(set(pulse2_doses.values())) == 1, "el modelo usa una dosis común para los tres lotes"
DELTA_N_KG_M3 = next(iter(pulse2_doses.values()))
GRID_H = 0.25
CO2_FACTOR_G_L_H_PER_SCCM = 44.0095 * 60.0 * 0.74 / (1000.0 * 24.16 * 2.0)
assert np.isclose(CO2_FACTOR_G_L_H_PER_SCCM, 0.0404391928807947)
print("ROOT:", ROOT)
print("t0:", T0, "| pulse 2 [h]:", PULSE2_H)

pd.set_option("display.max_rows", 120)


In [ ]:
THETA_FULL_PATH = LAB2026 / "results" / "estimability_historical_natural" / "theta_natural_full.csv"
THETA_AUX_PATH = LAB2026 / "results" / "final_operational_doe_v2" / "theta_final_operational_v2.csv"
CO2_PARAM_PATH = LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "three_state_co2_parameters.csv"

theta_table = pd.read_csv(THETA_FULL_PATH)
theta_full = dict(zip(theta_table["parameter"], theta_table["theta"].astype(float)))
assert set(base.FULL17) == set(theta_full), (set(base.FULL17) - set(theta_full), set(theta_full) - set(base.FULL17))

theta_aux = pd.read_csv(THETA_AUX_PATH, index_col=0).iloc[:, 0].astype(float).to_dict()
theta_model = {**theta_aux, **theta_full}  # theta_full siempre prevalece en la capa cinética

co2_parameter_table = pd.read_csv(CO2_PARAM_PATH)
co2_parameter_table = co2_parameter_table[
    co2_parameter_table["state"].eq("C_full_theta_refit")
    & co2_parameter_table["calibration_matrix"].eq("natural")
].copy()
co2_params = dict(zip(co2_parameter_table["parameter"], co2_parameter_table["estimate"].astype(float)))
assert set(co2_cross.PARAMETER_BOUNDS).issubset(co2_params)



display(theta_table)
display(co2_parameter_table[["parameter", "estimate", "active_bound"]])


## Parámetros editables

`nutrition_mode`: `actual`, `rampa_sin_atenuacion` o `directa`. `oxygen_enabled=False` fija $f_{ferm}=1$, $q_{resp}=0$: no representa una medición de O2 cero. `pulse_t_rise_h` es la duración de la rampa; `pulse_activity_gain` su multiplicador final de actividad, sólo en modo `actual`.

`O2_qmax_mg_gdw_h` y `O2_initial_scale` controlan consumo e inventario inicial de O2. `activation_enabled` y `activation_shift_h` permiten desactivar o desplazar $a(t)$. `kCO2_release_h` (h⁻¹), `CO2sat_scale` y `matrix_gain` controlan el pool y la escala de flujo. `A`, `tau_rise_h` y `tau_decay_h` controlan intensidad, subida y decaimiento del transitorio.

**Sólo manual:** `release_temperature_sensitivity_K` explora $k(T)=k_{ref}\exp[B(1/T_{ref}-1/T)]$; $B=0$ conserva $k$ constante. Temperaturas en kelvin; $B$ tiene unidad K y equivale a $E_{rel}/R$. No modifica `Eafe` ni la química. Horas/dosis de nutrición permanecen en sus registros.

In [ ]:
AUDIT_CONFIG = {
    'run_refits': True,
    'historical_weight': 1.0, 'recent_weight': 2.0,
    'max_nfev': 350,
    'multistart': ((0.10, 1.0, 4.0), (0.30, 2.0, 4.0),
                   (0.30, 4.0, 12.0), (0.60, 2.0, 8.0)),
    'bounds': {
        'kCO2_release_h': tuple(co2_cross.PARAMETER_BOUNDS['kCO2_release_h']),
        'CO2sat_scale': tuple(co2_cross.PARAMETER_BOUNDS['CO2sat_scale']),
        'matrix_gain': tuple(co2_cross.PARAMETER_BOUNDS['matrix_gain']),
        'A': (0.0, 1.5), 'tau_rise_h': (0.25, 12.0), 'tau_decay_h': (0.5, 24.0),
    },
}
# EDITAR AQUÍ. Después ejecutar esta celda y «Explorador manual».
MANUAL = {
    'labs': ('LAB019', 'LAB020', 'LAB021'),
    'nutrition_mode': 'actual',  # actual / rampa_sin_atenuacion / directa
    'oxygen_enabled': True,
    'activation_enabled': True,
    'activation_shift_h': 0.0,  # positivo retrasa la activación, en horas
    'pulse_t_rise_h': 56.741847,
    'pulse_activity_gain': 0.250011,
    'O2_qmax_mg_gdw_h': 0.253137,
    'O2_initial_scale': 0.179606,
    'kCO2_release_h': 0.09696,
    'CO2sat_scale': 2.5,
    'matrix_gain': 1.44670,
    'A': 1.5,                  # 0 desactiva el transitorio
    'tau_rise_h': 0.53733,
    'tau_decay_h': 4.40489,
    'release_temperature_sensitivity_K': 0.0,
    'release_reference_temperature_C': 20.0,
}
FROZEN_POOL_PARAMETERS = {
    'kCO2_release_h': co2_params['kCO2_release_h'],
    'CO2sat_scale': co2_params['CO2sat_scale'],
    'matrix_gain': co2_params['matrix_gain'],
    'A': 0.0, 'tau_rise_h': 2.0, 'tau_decay_h': 4.0,
    'release_temperature_sensitivity_K': 0.0,
    'release_reference_temperature_C': 20.0,
}
assert AUDIT_CONFIG['historical_weight'] > 0 and AUDIT_CONFIG['recent_weight'] > 0


In [ ]:
import hashlib

PROTECTED_PATHS = [
    LAB2026 / "notebooks" / "holdouts" / "lab019_021_theta_full_holdout.ipynb",
    LAB2026 / "notebooks" / "holdouts" / "lab019_021_theta_full_holdout.executed.ipynb",
    THETA_FULL_PATH,
    CO2_PARAM_PATH,
    REFERENCE_RESULTS_DIR / "co2_metrics.csv",
    REFERENCE_RESULTS_DIR / "co2_prediction_rows.csv",
    REFERENCE_RESULTS_DIR / "protocol_lock.json",
    DATA_016_018_DIR / "LAB016-018-offline-measurements.xlsx",
    LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "external_kinetic_holdout_initial_conditions_LAB013_015.csv",
    LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "external_holdout_prediction_rows_A_vs_B.csv",
    LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "external_holdout_support_LAB016_018.csv",
    LAB2026 / "results" / "lot2_data_preview" / "figures" / "lot2_F1_process_preview.png",
    LAB2026 / "results" / "lot2_data_preview" / "figures" / "lot2_F2_process_preview.png",
    LAB2026 / "results" / "lot2_data_preview" / "figures" / "lot2_F3_process_preview.png",
]

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

PROTECTED_PATHS.extend([
    LAB2026 / 'notebooks/diagnostics/lab019_021_co2_onset_effective_model_test.ipynb',
    LAB2026 / 'notebooks/diagnostics/lab019_021_co2_onset_effective_model_test.executed.ipynb',
])
frozen_theta_snapshot = dict(theta_full)
frozen_co2_snapshot = dict(co2_params)
protected_before = {str(path): sha256(path) for path in PROTECTED_PATHS}
print('Artefactos protegidos:', len(PROTECTED_PATHS))


## Datos, máscaras y eventos

Preparación independiente reproducida desde el diagnóstico anterior: química actual y correcciones de m12, ICs, temperatura medida, cero del sensor, máscara de agitación inicial y censura izquierda. No se ejecuta ni se modifica otro notebook.

LAB019–021: dosis de 80 mg/L registrada en `Nutricion`. Retiros y 100 mL añadidos se documentan separados; no se supone volumen real constante ni se reconstruye YAN desde gramos. El flujo conserva la normalización histórica a 2 L; esta auditoría no introduce balances de volumen/dilución.

LAB016–018: biomasa heredada de LAB013–015 (mismo mosto), demás ICs del Excel y horas de nutrición del logger. Dosis de referencia **0.14 kg/m³ por evento**, pendiente de confirmar. LAB017 tiene dos eventos: cinética y transitorio conservan ambos. Su rampa histórica usa primera hora y diferencia total con/sin nutrición, como en la referencia; se conserva esa limitación al comparar bloques.

In [ ]:
def read_csv_window(path, usecols, start, end, chunksize=500_000):
    frames = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False):
        stamp = pd.to_datetime(chunk["timestamp"], errors="coerce")
        keep = stamp.between(start, end)
        if keep.any():
            part = chunk.loc[keep].copy()
            part["timestamp"] = stamp.loc[keep]
            frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=usecols)


window_start, window_end = T0 - pd.Timedelta(hours=6), T0 + pd.Timedelta(hours=170)
temperature_raw = read_csv_window(
    DATA_DIR / "backup_global_temperatura.csv", ["timestamp", "fermentador", "T", "SP"], window_start, window_end
)
for col in ("T", "SP"):
    temperature_raw[col] = pd.to_numeric(temperature_raw[col], errors="coerce")
co2_raw = read_csv_window(
    DATA_DIR / "backup_global_co2.csv", ["timestamp", "fermentador", "flow_sccm", "flow_filt_sccm", "status"], window_start, window_end
)
for col in ("flow_sccm", "flow_filt_sccm"):
    co2_raw[col] = pd.to_numeric(co2_raw[col], errors="coerce")

TEMP_RESAMPLE = "1min"    # mediana por minuto: elimina outliers y uniformiza el cadence
TEMP_MEDIAN = "20min"     # mediana móvil: suprime ráfagas y spikes del control on/off
TEMP_MEAN = "45min"       # media móvil: suaviza la oscilación de banda del control

temp_log = {}
for _lab in LABS:
    _f = FERMENTER[_lab]
    frame = (temperature_raw[temperature_raw["fermentador"].eq(_f)]
             .dropna(subset=["T"]).sort_values("timestamp")
             .set_index("timestamp")[["T", "SP"]])
    frame = frame.resample(TEMP_RESAMPLE).median()
    t_med = frame["T"].rolling(TEMP_MEDIAN, center=True, min_periods=1).median()
    temp_log[_f] = pd.DataFrame({
        "T_filt": t_med.rolling(TEMP_MEAN, center=True, min_periods=1).mean(),
        "SP": frame["SP"],
    })


In [ ]:
from shared.new_must_data_loader import ETHANOL_G_L_PER_PERCENT_VV


def numeric(value):
    try:
        out = float(value)
        return out if np.isfinite(out) else np.nan
    except (TypeError, ValueError):
        return np.nan


def load_offline_samples():
    path = DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx"
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = []
    for excel_row, row in enumerate(wb["MedicionesOffline"].iter_rows(min_row=2, values_only=True), start=2):
        fecha, hora, lab, brix, density = row[0], row[1], row[2], row[5], row[6]
        if lab is None and brix is None and density is None:
            continue
        if lab not in LABS or fecha is None or hora is None or row[4] is None:
            raise ValueError(f"MedicionesOffline fila {excel_row}: metadatos de muestra incompletos")
        hh = hora.time() if isinstance(hora, datetime) else hora
        rows.append({
            "lab": str(lab), "sample": str(row[4]).strip(),
            "sample_timestamp": datetime.combine(fecha.date() if hasattr(fecha, "date") else fecha, hh),
            "Brix": numeric(brix), "density": numeric(density),
            "Etanol_pct_vv": numeric(row[10]), "metadata_fallback": False,
        })
    wb.close()
    result = pd.DataFrame(rows)
    assert not result.duplicated(["lab", "sample"]).any(), "muestras offline duplicadas"
    return result

def load_y15():
    rows = []
    paths = sorted(DATA_DIR.glob("EXPAuto*.txt")) + sorted(DATA_DIR.glob("ONLINE*.txt"))
    for path in paths:
        for line in open(path, encoding="utf-8-sig", errors="replace"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 6:
                continue
            raw = fields[3].strip()
            value = float(raw.replace(",", ".")) if re.fullmatch(r"-?\d+(,\d+)?", raw) else np.nan
            rows.append({"sample_id": fields[0], "analyte": fields[1], "value": value, "raw_value": raw, "analysis_timestamp": fields[5], "source_file": path.name})
    long = pd.DataFrame(rows).drop_duplicates(["sample_id", "analyte"], keep="first")
    wide = long.pivot(index="sample_id", columns="analyte", values="value")
    raw_wide = long.pivot(index="sample_id", columns="analyte", values="raw_value")
    return long, wide, raw_wide


offline = load_offline_samples()
# Dilución Alcolyzer m12, distinta por lote (cuaderno de laboratorio): LAB020 27.5 mL de vino + 10 mL de agua;
# LAB021 32.5 mL de vino + 5 mL de agua (ambas a 37.5 mL). Sólo corrige etanol (medido en la mezcla);
# Brix y densidad se midieron antes de diluir.
DILUTION_FACTORS_M12 = {"LAB020": 37.5 / 27.5, "LAB021": 37.5 / 32.5}
for _lab_m12, _factor in DILUTION_FACTORS_M12.items():
    _sel = offline["lab"].eq(_lab_m12) & offline["sample"].eq("12")
    offline.loc[_sel, "Etanol_pct_vv"] = offline.loc[_sel, "Etanol_pct_vv"] * _factor
assert np.isclose(offline.loc[(offline["lab"] == "LAB020") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 7.33 * 15 / 11)
assert np.isclose(offline.loc[(offline["lab"] == "LAB021") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 8.66 * 15 / 13)
y15_long, y15, y15_raw = load_y15()

chem_rows = []
for lab in LABS:
    samples = offline[offline["lab"].eq(lab)].copy().set_index("sample")
    numbered_samples = sorted((item for item in samples.index if str(item).isdigit()), key=int)
    assert "1" in numbered_samples, f"{lab}: falta muestra inicial"
    for sample in numbered_samples:
        sid = f"{lab}-{sample}"
        meta = samples.loc[sample]
        stamp = pd.Timestamp(meta["sample_timestamp"])
        sample_t_h = (stamp - T0).total_seconds() / 3600.0
        t_h = 0.0 if sample == "1" else sample_t_h
        get = lambda name: float(y15.loc[sid, name]) if sid in y15.index and name in y15.columns and pd.notna(y15.loc[sid, name]) else np.nan
        get_raw = lambda name: str(y15_raw.loc[sid, name]) if sid in y15_raw.index and name in y15_raw.columns and pd.notna(y15_raw.loc[sid, name]) else ""
        gf_320_raw = get("GLU-FRU-320")
        gf_low_raw = get("GLUCOSE-FRUCTOSE")
        if np.isfinite(gf_320_raw) and gf_320_raw >= 0.0:
            gf, gf_source = gf_320_raw, "GLU-FRU-320"
        elif np.isfinite(gf_low_raw) and gf_low_raw >= 0.0:
            gf, gf_source = gf_low_raw, "GLUCOSE-FRUCTOSE"
        else:
            gf, gf_source = np.nan, ""
        glucose_320_raw = get("GLUCOSE-320")
        glucose_low_raw = get("GLUCOSE")
        glucose_raw = glucose_320_raw if np.isfinite(glucose_320_raw) else glucose_low_raw
        glucose = glucose_raw if np.isfinite(glucose_raw) and glucose_raw >= 0.0 else np.nan
        fructose = gf - glucose if np.isfinite(gf) and np.isfinite(glucose) and gf >= glucose else np.nan
        pan, ammonia_raw = get("PAN"), get("AMMONIA")
        ammonia = max(ammonia_raw, 0.0) if np.isfinite(ammonia_raw) else np.nan
        yan = pan + 0.823 * ammonia if np.isfinite(pan) and np.isfinite(ammonia) else np.nan
        chem_rows.append({
            "lab": lab, "sample": sample, "sample_timestamp": stamp, "sample_t_h": sample_t_h, "t_h": t_h,
            "metadata_fallback": bool(meta["metadata_fallback"]), "density": float(meta["density"]), "Brix": float(meta["Brix"]),
            "GF_g_l": gf, "GF_source": gf_source,
            "GF_320_raw_g_l": gf_320_raw, "GF_low_raw_g_l": gf_low_raw,
            "GF_320_below_zero": bool(np.isfinite(gf_320_raw) and gf_320_raw < 0.0),
            "G_g_l": glucose, "G_raw_g_l": glucose_raw, "F_g_l": fructose,
            "PAN_mg_l": pan, "NH3_raw_mg_l": ammonia_raw, "NH3_censored_mg_l": ammonia,
            "YAN_mg_l": yan, "N_kg_m3": yan / 1000.0 if np.isfinite(yan) else np.nan,
            "Gly_g_l": get("GLYCEROL"), "Pyr_mg_l": get("PYRUVIC ACID"),
            "Etanol_pct_vv": float(meta["Etanol_pct_vv"]),
            "E_g_l": meta["Etanol_pct_vv"] * ETHANOL_G_L_PER_PERCENT_VV if np.isfinite(meta["Etanol_pct_vv"]) else np.nan,
            "AcAld_mg_l": get("ACETALDEHID-1000"), "Acetate_g_l": get("ACETIC ACID"),
            "G_raw_text": get_raw("GLUCOSE-320") or get_raw("GLUCOSE"),
            "GF_raw_text": get_raw("GLU-FRU-320"),
            "GF_low_raw_text": get_raw("GLUCOSE-FRUCTOSE"),
            "NH3_below_zero": bool(np.isfinite(ammonia_raw) and ammonia_raw < 0.0),
        })
chem = pd.DataFrame(chem_rows).sort_values(["lab", "t_h"]).reset_index(drop=True)
chem["is_initial"] = chem["sample"].eq("1")
chem["soft_flag_GF"] = chem["lab"].eq("LAB019") & chem["sample"].eq("4")
chem["soft_flag_metadata"] = chem["metadata_fallback"]
chem["y15_available"] = chem[["GF_g_l", "YAN_mg_l", "Gly_g_l", "Pyr_mg_l", "AcAld_mg_l"]].notna().any(axis=1)
assert chem[chem["sample"].eq("1")]["E_g_l"].notna().all(), "falta Etanol m1 (inicializa E0)"

assert np.allclose(chem[chem["sample"].eq("1")]["t_h"], 0.0)
assert chem.query("lab == 'LAB020' and sample == '4'")["G_g_l"].isna().all()
assert chem.query("lab == 'LAB020' and sample == '4'")["G_raw_g_l"].iloc[0] == -7.28
assert chem[chem["sample"].eq("12")]["y15_available"].all(), "falta exportación Y15 m12"
assert chem.loc[chem["sample"].eq("12"), "GF_320_below_zero"].all(), "revisar QA de GF m12"
display(chem[["lab", "sample", "sample_t_h", "t_h", "GF_g_l", "G_g_l", "F_g_l", "YAN_mg_l", "Gly_g_l", "E_g_l", "Pyr_mg_l", "AcAld_mg_l", "y15_available", "NH3_below_zero", "metadata_fallback"]])
print("Muestras sin resultado Y15:", chem.loc[~chem["y15_available"], ["lab", "sample"]].to_dict("records"))


In [ ]:
historical_data = natural_loader.load_historical_medium_data("natural")
historical_batches = {b.batch: b for b in joint.make_secondary_batches(historical_data)}
historical_secondary_initials = joint.secondary_initials_by_medium(historical_data)["natural"]

def temperature_profile(lab, grid):
    f = FERMENTER[lab]
    frame = temperature_raw[temperature_raw["fermentador"].eq(f)].dropna(subset=["T"]).sort_values("timestamp")
    t = (frame["timestamp"] - T0).dt.total_seconds().to_numpy() / 3600.0
    return np.interp(grid, t, frame["T"].to_numpy(dtype=float))


holdout_batches = {}
core_predictions = {}
secondary_predictions = {}
initial_rows = []

# Biomasa inicial desde el m1 de Oculyze de cada réplica (~25 min tras la aplicación de
# levadura), con la misma conversión del pipeline histórico (30 pg/célula). La convención
# primaria es Concentración cruda x 0.03, idéntica a la que ancló las X de los originales
# (p.ej. LAB009 t=0: 12.04 x 0.03 = 0.361); la variante con CERO restado se reporta como referencia.
from shared.new_must_data_loader import MILLION_CELLS_ML_TO_KG_M3
assert abs(MILLION_CELLS_ML_TO_KG_M3 - 0.03) < 1e-12

def oculyze_m1_biomass(lab):
    with open(DATA_DIR / f"Oculyze_report_{lab}.csv", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    m1 = next(r for r in rows if r["Name"].strip().upper() == f"{lab}-1")
    cero = next((r for r in rows if r["Name"].strip().upper() == "CERO"), None)
    concentration = float(m1["Concentration"])
    viability = float(m1["Viability"])
    return {
        "x0_kg_m3": concentration * MILLION_CELLS_ML_TO_KG_M3,
        "xd0_kg_m3": concentration * MILLION_CELLS_ML_TO_KG_M3 * (1.0 - viability / 100.0),
        "oculyze_m1_concentration_mcells_ml": concentration,
        "oculyze_m1_viability_pct": viability,
        "x0_cero_subtracted_kg_m3": (
            max(concentration - float(cero["Concentration"]), 0.0) * MILLION_CELLS_ML_TO_KG_M3
            if cero is not None else float("nan")
        ),
    }

oculyze_biomass = {lab: oculyze_m1_biomass(lab) for lab in LABS}
for lab in LABS:
    c = chem[chem["lab"].eq(lab)].sort_values("t_h")
    init = c[c["sample"].eq("1")].iloc[0]
    original = historical_batches[MATCHED_ORIGINAL[lab]]
    f = FERMENTER[lab]
    co2_lab = co2_raw[co2_raw["fermentador"].eq(f)]
    logger_horizon = min(170.0, float((co2_lab["timestamp"].max() - T0).total_seconds() / 3600.0))
    measured_states = ["GF_g_l", "YAN_mg_l", "Gly_g_l", "E_g_l", "Pyr_mg_l", "AcAld_mg_l", "Acetate_g_l"]
    scored_chemistry = c[c[measured_states].notna().any(axis=1)]
    chemistry_horizon = float(scored_chemistry["t_h"].max())
    horizon = max(logger_horizon, chemistry_horizon)
    temp_lab = temperature_raw[temperature_raw["fermentador"].eq(f)].dropna(subset=["T"])
    temperature_last_h = float((temp_lab["timestamp"].max() - T0).total_seconds() / 3600.0)
    grid = np.arange(0.0, horizon + GRID_H / 2.0, GRID_H)
    grid = np.unique(np.r_[grid, c["t_h"].to_numpy(dtype=float), PULSE2_H[lab]])
    grid = grid[(grid >= 0.0) & (grid <= horizon + 1e-9)]
    temp = temperature_profile(lab, grid)
    # E0 desde el Etanol de m1 (% v/v -> g/L); los originales también inicializan E con su ETANOL t=0.
    initials = {
        "X": float(oculyze_biomass[lab]["x0_kg_m3"]), "Xd": float(oculyze_biomass[lab]["xd0_kg_m3"]),
        "N": float(init["N_kg_m3"]), "G": float(init["G_g_l"]), "F": float(init["F_g_l"]),
        "E": float(init["E_g_l"]), "Gly": float(init["Gly_g_l"]),
        "Pyr": float(init["Pyr_mg_l"]),
        "AcAld": float(historical_secondary_initials["AcAld"]),
        "Acetate": float(init["Acetate_g_l"]),
        "O2": float(historical_secondary_initials["O2"]), "CO2": 0.0,
    }
    observations = {state: np.full(len(grid), np.nan) for state in base.STATE_NAMES}
    observations.update({state: np.full(len(grid), np.nan) for state in ("Pyr", "AcAld", "Acetate", "O2")})
    # Química observada sólo se almacena para el diagnóstico condicionado; no altera la simulación cinética.
    for row in c.itertuples():
        idx = int(np.argmin(np.abs(grid - float(row.t_h))))
        if abs(grid[idx] - float(row.t_h)) > 1e-7:
            continue
        for state, value in (("G", row.G_g_l), ("F", row.F_g_l), ("N", row.N_kg_m3), ("Gly", row.Gly_g_l), ("E", row.E_g_l),
        ("Pyr", row.Pyr_mg_l), ("AcAld", row.AcAld_mg_l), ("Acetate", row.Acetate_g_l)):
            observations[state][idx] = value
    batch = base.BatchData(
        medium="natural", batch=lab, time=grid, temperature_c=temp,
        pulses={"N": ((PULSE2_H[lab], DELTA_N_KG_M3),), "G": tuple(), "F": tuple(), "E": tuple(), "X": tuple()},
        observations=observations, initials=initials,
    )
    core = base.simulate(batch, theta_full, grid)
    assert core is not None and np.isfinite(core.to_numpy()).all()
    secondary = secondary_v2.integrate_secondary_v2(batch, theta_model, core)
    assert np.isfinite(secondary.to_numpy()).all()
    holdout_batches[lab], core_predictions[lab], secondary_predictions[lab] = batch, core, secondary
    initial_rows.append({
        "lab": lab, "x0_source": f"Oculyze m1 {lab}-1", **oculyze_biomass[lab],
        "x0_heredado_del_original_kg_m3": float(original.initials["X"]),
        "matched_original_reference": MATCHED_ORIGINAL[lab],
        **initials, "horizon_h": horizon,
        "temperature_extrapolated_h": max(0.0, horizon - temperature_last_h),
    })

initial_table = pd.DataFrame(initial_rows)
display(initial_table)
if initial_table["temperature_extrapolated_h"].gt(0).any():
    print("AVISO: temperatura prolongada con el último valor medido hasta la última química; interpretar los puntos posteriores al logger como sensibilidad.")


In [ ]:
co2_hourly_parts, mask_rows = [], []
for lab in LABS:
    f = FERMENTER[lab]
    frame = co2_raw[co2_raw["fermentador"].eq(f)].sort_values("timestamp").copy()
    frame["t_h"] = (frame["timestamp"] - T0).dt.total_seconds() / 3600.0
    baseline = frame[frame["t_h"].between(-2.0, -0.5)]["flow_sccm"].dropna().to_numpy(dtype=float)
    baseline_center = float(np.median(baseline))
    sigma = float(1.4826 * np.median(np.abs(baseline - baseline_center)))
    threshold = max(3.5 * sigma, 0.5)
    search = frame[frame["t_h"].between(-0.5, 0.75)].copy()
    crossing = search[np.abs(search["flow_sccm"] - baseline_center) > threshold]
    if crossing.empty:
        event_start_h, event_raw_end_h = 0.0, 1.0
    else:
        event_start_h, event_raw_end_h = float(crossing["t_h"].min()), float(crossing["t_h"].max())
    mask_end_h = max(0.0, event_raw_end_h) + 1.5

    zero_support = frame[frame["t_h"].between(mask_end_h, 12.0)]["flow_filt_sccm"].dropna()
    zero_sccm = max(0.0, float(zero_support.quantile(0.10))) if len(zero_support) else 0.0
    frame["flow_corrected_sccm"] = np.maximum(frame["flow_filt_sccm"] - zero_sccm, 0.0)
    hourly = frame.set_index("timestamp")[["flow_corrected_sccm", "flow_filt_sccm", "flow_sccm"]].resample("1h").median().dropna(subset=["flow_corrected_sccm"]).reset_index()
    hourly["t_h"] = (hourly["timestamp"] - T0).dt.total_seconds() / 3600.0
    hourly = hourly[hourly["t_h"].between(0.0, 170.0)].copy()
    hourly["lab"] = lab
    hourly["co2_observed_g_l_h"] = hourly["flow_corrected_sccm"] * CO2_FACTOR_G_L_H_PER_SCCM
    hourly["co2_valid"] = hourly["t_h"] > mask_end_h
    hourly["left_censored"] = hourly["co2_observed_g_l_h"] < co2_cross.CO2_DETECTION_LIMIT_G_L_H
    co2_hourly_parts.append(hourly)
    mask_rows.append({
        "lab": lab, "fermenter": f, "baseline_raw_sccm": baseline_center, "baseline_sigma_sccm": sigma,
        "artifact_threshold_sccm": threshold, "event_start_h": event_start_h,
        "event_raw_end_h": event_raw_end_h, "recovery_tail_min": 90, "mask_end_h": mask_end_h,
        "sensor_zero_sccm": zero_sccm,
    })

co2_hourly = pd.concat(co2_hourly_parts, ignore_index=True)
mask_audit = pd.DataFrame(mask_rows)
display(mask_audit.round(4))


In [ ]:
def activity_bracket(time_h, sugar, ethanol):
    time_h = np.asarray(time_h, dtype=float)
    sugar = np.asarray(sugar, dtype=float)
    ethanol = np.asarray(ethanol, dtype=float)
    valid = np.isfinite(sugar) | np.isfinite(ethanol)
    indices = np.flatnonzero(valid)
    if not len(indices):
        return {"lower_h": 0.0, "upper_h": 0.0, "center_h": 0.0, "scale_h": 1.0}
    s0 = sugar[np.isfinite(sugar)][0] if np.isfinite(sugar).any() else np.nan
    e0 = ethanol[np.isfinite(ethanol)][0] if np.isfinite(ethanol).any() else np.nan
    active = (np.isfinite(sugar) & (sugar <= s0 - 5.0)) | (np.isfinite(ethanol) & (ethanol >= e0 + 2.0))
    active_indices = np.flatnonzero(active)
    if len(active_indices):
        upper_i = int(active_indices[0])
        earlier = indices[indices < upper_i]
        lower_i = int(earlier[-1]) if len(earlier) else upper_i
        lower, upper = float(time_h[lower_i]), float(time_h[upper_i])
    else:
        lower = upper = float(time_h[indices[0]])
    return {"lower_h": lower, "upper_h": upper, "center_h": 0.5 * (lower + upper), "scale_h": max(1.0, (upper - lower) / (2.0 * math.log(9.0))) if upper > lower else 1.0}


def sustained_onset(time_h, values, threshold, consecutive=3):
    t, y = np.asarray(time_h, float), np.asarray(values, float)
    above = np.isfinite(y) & (y >= threshold)
    for i in range(0, len(y) - consecutive + 1):
        if above[i:i + consecutive].all():
            return float(t[i])
    return np.nan


def onset_threshold(group, observed_column="co2_rate_g_l_h"):
    early = group[group["t_h"] <= 12.0]
    baseline = float(early[observed_column].median()) if len(early) else 0.0
    peak = float(group[observed_column].max())
    detection = (
        float(group["detection_limit_g_l_h"].median())
        if "detection_limit_g_l_h" in group else co2_cross.CO2_DETECTION_LIMIT_G_L_H
    )
    return max(detection, baseline + 0.10 * max(peak - baseline, 0.0))


def trapezoid_on_contiguous_support(time_h, values):
    time_h = np.asarray(time_h, dtype=float)
    values = np.asarray(values, dtype=float)
    dt = np.diff(time_h)
    keep = (dt > 0.0) & (dt <= 1.5)
    return float(np.sum(0.5 * (values[:-1] + values[1:]) * dt * keep))


def metric_row(model, lab, group, predicted):
    observed = (
        group["co2_observed_g_l_h"].to_numpy(dtype=float)
        if "co2_observed_g_l_h" in group else group["co2_rate_g_l_h"].to_numpy(dtype=float)
    )
    time_h = group["t_h"].to_numpy(dtype=float) if "t_h" in group else group["time_h"].to_numpy(dtype=float)
    threshold_group = group.rename(columns={"co2_observed_g_l_h": "co2_rate_g_l_h", "time_h": "t_h"})
    threshold = onset_threshold(threshold_group)
    onset_obs = sustained_onset(time_h, observed, threshold)
    onset_pred = sustained_onset(time_h, predicted, threshold)
    error = predicted - observed
    peak_obs = float(np.max(observed))
    peak_pred = float(np.max(predicted))
    auc_obs = trapezoid_on_contiguous_support(time_h, observed)
    auc_pred = trapezoid_on_contiguous_support(time_h, predicted)
    return {
        "model": model, "lab": lab, "n": len(group),
        "RMSE_g_l_h": float(np.sqrt(np.mean(error**2))),
        "NRMSE_peak": float(np.sqrt(np.mean(error**2)) / max(peak_obs, 1e-12)),
        "onset_threshold_g_l_h": threshold,
        "onset_observed_h": onset_obs, "onset_predicted_h": onset_pred,
        "onset_error_h": onset_pred - onset_obs if np.isfinite(onset_obs) and np.isfinite(onset_pred) else np.nan,
        "peak_observed_g_l_h": peak_obs, "peak_predicted_g_l_h": peak_pred,
        "peak_time_observed_h": float(time_h[np.argmax(observed)]),
        "peak_time_predicted_h": float(time_h[np.argmax(predicted)]),
        "AUC_observed_g_l": auc_obs, "AUC_predicted_g_l": auc_pred,
        "AUC_relative_error": (auc_pred - auc_obs) / max(auc_obs, 1e-12),
    }


def complete_observed_curve(lab):
    if lab in RECENT_TRAIN_LABS:
        group = co2_hourly[
            co2_hourly["lab"].eq(lab) & co2_hourly["co2_valid"]
        ].sort_values("t_h")
        return group["t_h"].to_numpy(dtype=float), group["co2_observed_g_l_h"].to_numpy(dtype=float)
    group = historical_obs[historical_obs["batch"].eq(lab)].sort_values("t_h")
    return group["t_h"].to_numpy(dtype=float), group["co2_rate_g_l_h"].to_numpy(dtype=float)


In [ ]:
from scipy.optimize import least_squares

HISTORICAL_CO2_DIR = (
    LAB2026 / "results" /
    "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010"
)
historical_obs = pd.read_csv(HISTORICAL_CO2_DIR / "co2_observations_hourly.csv")
historical_obs = historical_obs[historical_obs["matrix"].eq("natural")].copy()
for column in ("calibratable", "left_censored"):
    historical_obs[column] = historical_obs[column].map(
        lambda value: value if isinstance(value, bool) else str(value).strip().lower() == "true"
    )

historical_chemistry, historical_natural_batches, _, historical_natural_tables = (
    natural_loader.make_medium_batches("natural")
)
historical_schedule = co2_cross.natural_nutrient_pulse_schedule(
    historical_chemistry, historical_natural_tables["metadata"]
)
historical_natural_batches = co2_cross.override_natural_nutrient_pulses(
    historical_natural_batches, historical_schedule
)
historical_batches_all = {batch.batch: batch for batch in historical_natural_batches}
historical_cache, historical_driver_audit = co2_cross.build_driver_cache(
    historical_batches_all,
    {"natural": theta_full},
    historical_obs,
)

HISTORICAL_TRAIN_LABS = ("LAB005", "LAB007", "LAB008")
RECENT_TRAIN_LABS = LABS
TRAIN_LABS = HISTORICAL_TRAIN_LABS + RECENT_TRAIN_LABS
VALIDATION_LABS = ("LAB011", "LAB012")
SENSITIVITY_LABS = ("LAB010",)
FIT_LAB_WEIGHT = {
    **{lab: AUDIT_CONFIG['historical_weight'] for lab in HISTORICAL_TRAIN_LABS},
    **{lab: AUDIT_CONFIG['recent_weight'] for lab in RECENT_TRAIN_LABS},
}
assert "LAB004" not in TRAIN_LABS and "LAB006" not in TRAIN_LABS
assert set(HISTORICAL_TRAIN_LABS + VALIDATION_LABS + SENSITIVITY_LABS).issubset(historical_cache)


# Cache para LAB019–LAB021. La química observada no se usa como gate.
recent_cache = {}
for lab in RECENT_TRAIN_LABS:
    observed = co2_hourly[co2_hourly["lab"].eq(lab)].sort_values("t_h")
    horizon = float(observed["t_h"].max())
    stub = observed[["t_h"]].copy()
    stub["batch"] = lab
    stub["matrix"] = "natural"
    stub["calibratable"] = True
    stub["chemistry_last_h"] = horizon
    caches, _ = co2_cross.build_driver_cache(
        {lab: holdout_batches[lab]}, {"natural": theta_full}, stub
    )
    recent_cache[lab] = caches[lab]

# Holdout LAB016–LAB018: soporte observado ya auditado y limpio, sin usarlo en el fit.
EXTERNAL_LABS = ("LAB016", "LAB017", "LAB018")
external_rows_all = pd.read_csv(HISTORICAL_CO2_DIR / "external_holdout_prediction_rows_A_vs_B.csv")
external_observed = (
    external_rows_all[["batch", "time_h", "observed_g_l_h"]]
    .drop_duplicates()
    .rename(columns={"batch": "lab", "observed_g_l_h": "co2_observed_g_l_h"})
    .sort_values(["lab", "time_h"])
    .reset_index(drop=True)
)
assert set(external_observed["lab"]) == set(EXTERNAL_LABS)
external_support_audit = pd.read_csv(HISTORICAL_CO2_DIR / "external_holdout_support_LAB016_018.csv")

external_initial_reference = pd.read_csv(
    HISTORICAL_CO2_DIR / "external_kinetic_holdout_initial_conditions_LAB013_015.csv"
)
inherited_x0 = float(external_initial_reference["X"].mean())
inherited_xd0 = float(external_initial_reference["Xd"].mean())
external_initial_sheet = pd.read_excel(
    DATA_016_018_DIR / "LAB016-018-offline-measurements.xlsx",
    sheet_name="CondicionesIniciales",
)
external_initial_sheet = external_initial_sheet.set_index("CodigoLab")

external_temp_files = {
    "LAB016": "Temp_F1_LAB016.csv",
    "LAB017": "Temp_F2_LAB017.csv",
    "LAB018": "Temp_F3._LAB018.csv",
}
external_co2_files = {
    "LAB016": "CO2_FILT_F1_LAB016.csv",
    "LAB017": "CO2_FILT_F2_LAB017.csv",
    "LAB018": "CO2_FILT_F3_LAB018.csv",
}
external_batches = {}
external_cache = {}
external_initial_rows = []
external_full_observed_parts = []
external_pulse_rows, external_mask_rows = [], []
external_nutrition_schedules = {}
external_reference_dose_kg_m3 = 0.14  # mismo supuesto de la prueba previa; no una medición de YAN


def logged_pump_events(lab, frame, t0):
    state = pd.to_numeric(frame['nutricion_activa'], errors='raise')
    assert state.notna().all() and set(state.unique()).issubset({0, 1}), lab
    starts = np.flatnonzero((state.eq(1) & ~state.shift(fill_value=0).eq(1)).to_numpy())
    ends = np.flatnonzero((state.eq(0) & state.shift(fill_value=0).eq(1)).to_numpy())
    rows = []
    for start in starts:
        after = ends[ends > start]
        assert len(after), f'{lab}: pulso sin flanco de cierre'
        end = int(after[0])
        stamp_start, stamp_end = frame.iloc[start]['timestamp'], frame.iloc[end]['timestamp']
        rows.append({'lab': lab, 'start': stamp_start, 'end': stamp_end,
                     'start_h': (stamp_start-t0).total_seconds()/3600,
                     'end_h': (stamp_end-t0).total_seconds()/3600,
                     'duration_min': (stamp_end-stamp_start).total_seconds()/60})
    return rows


def logged_co2_artifact_windows(lab, events, t0):
    filename = external_co2_files[lab].replace('CO2_FILT_', 'CO2_')
    raw = pd.read_csv(DATA_016_018_DIR / filename, parse_dates=['timestamp']).sort_values('timestamp')
    times = raw['timestamp'].astype('int64').to_numpy()
    values = raw['flow_sccm'].to_numpy(float)
    windows = []
    for event in events:
        start_ns, end_ns = pd.Timestamp(event['start']).value, pd.Timestamp(event['end']).value
        p0, p1 = np.searchsorted(times, [start_ns-int(1200e9), start_ns])
        e0, e1 = np.searchsorted(times, [start_ns, end_ns+int(120e9)])
        pre, during = values[p0:p1], values[e0:e1]
        pre, during = pre[np.isfinite(pre)], during[np.isfinite(during)]
        if len(pre) < 3 or len(during) < 1:
            continue
        center = float(np.median(pre))
        sigma = 1.4826 * float(np.median(np.abs(pre-center)))
        threshold = max(3.5*sigma, 0.5)
        segment = values[e0:e1]
        crossings = np.flatnonzero(np.isfinite(segment) & (np.abs(segment-center)>threshold))
        if len(crossings):
            artifact_start = pd.Timestamp(times[e0+crossings[0]])
            artifact_end = event['end']+pd.Timedelta(minutes=90)
            windows.append((artifact_start,artifact_end))
            external_mask_rows.append({'lab':lab,'artifact_start_h':(artifact_start-t0).total_seconds()/3600,
                                       'artifact_end_h':(artifact_end-t0).total_seconds()/3600,
                                       'threshold_raw_sccm':threshold,'recovery_min':90})
    return windows


def cache_with_recorded_pulses(template, pulse_batch, pulse_core, schedule):
    # La cinética aplica ambos saltos exactos. El API histórico admite un solo pulso;
    # su rampa usa la primera hora y la suma. El kernel transitorio conserva cada evento.
    time = template.time_h
    production = co2_cross.co2_model.co2_production_g_l_h(theta_full,pulse_batch,pulse_core,time)
    biomass = np.maximum(pulse_core['X'].to_numpy(float),0)
    e,g,f = (pulse_core[state].to_numpy(float) for state in ('E','G','F'))
    temp = template.temperature_used_c
    csat = 1.69*np.exp(-0.032*(temp-20))*np.exp(0.0016*np.maximum(e,0))*np.exp(-0.0012*np.maximum(g+f,0))
    o2_sat = np.array([co2_cross.co2_model.o2_saturation_mg_l(float(t),float(ee),float(gg),float(ff),1.0)
                       for t,ee,gg,ff in zip(temp,e,g,f)])
    return replace(template,base_qprod_g_l_h=np.asarray(production,float),
                   n_pulse_qprod_increment_g_l_h=np.asarray(production,float)-template.base_qprod_no_n_pulse_g_l_h,
                   biomass_g_l=biomass,n_pulse_biomass_increment_g_l=biomass-template.biomass_no_n_pulse_g_l,
                   n_pulse_time_h=float(schedule[0][0]),n_pulse_amount_kg_m3=sum(a for _,a in schedule),
                   csat_base_g_l=csat,o2_saturation_base_mg_l=o2_sat)


for lab in EXTERNAL_LABS:
    init = external_initial_sheet.loc[lab]
    t0 = pd.Timestamp(init["Fecha"]).normalize() + pd.to_timedelta(str(init["Hora"]))
    support = external_observed[external_observed["lab"].eq(lab)].sort_values("time_h")
    strict_holdout_last_h = float(support["time_h"].max())

    full_co2 = pd.read_csv(DATA_016_018_DIR / external_co2_files[lab], parse_dates=["timestamp"])
    full_co2["t_h"] = (full_co2["timestamp"] - t0).dt.total_seconds() / 3600.0
    baseline_support = full_co2[full_co2["t_h"].between(-2.0, -0.5)]["flow_filt_sccm"].dropna()
    baseline_sccm = float(baseline_support.median()) if len(baseline_support) else 0.0
    full_co2["flow_corrected_sccm"] = np.maximum(full_co2["flow_filt_sccm"] - baseline_sccm, 0.0)
    full_hourly = (
        full_co2.set_index("timestamp")[["flow_corrected_sccm"]]
        .resample("1h").median().dropna().reset_index()
    )
    full_hourly["t_h"] = (full_hourly["timestamp"] - t0).dt.total_seconds() / 3600.0
    full_hourly = full_hourly[full_hourly["t_h"].ge(0.0)].copy()
    full_hourly["lab"] = lab
    full_hourly["co2_observed_g_l_h"] = full_hourly["flow_corrected_sccm"] * CO2_FACTOR_G_L_H_PER_SCCM

    horizon = float(full_hourly["t_h"].max())
    grid_end = math.ceil(horizon / GRID_H) * GRID_H
    grid = np.arange(0.0, grid_end + GRID_H / 2.0, GRID_H)

    temp = pd.read_csv(DATA_016_018_DIR / external_temp_files[lab], parse_dates=["timestamp"])
    temp["time_h"] = (temp["timestamp"] - t0).dt.total_seconds() / 3600.0
    temp = temp.sort_values('timestamp').drop_duplicates('timestamp').reset_index(drop=True)
    pump_events = logged_pump_events(lab,temp,t0)
    nutrition = [event for event in pump_events if event['start_h'] >= 48.0]
    assert len(nutrition) == (2 if lab=='LAB017' else 1), (lab,nutrition)
    schedule = tuple((event['start_h'],external_reference_dose_kg_m3) for event in nutrition)
    external_nutrition_schedules[lab] = schedule
    for event in nutrition:
        external_pulse_rows.append({**event,'delta_N_kg_m3_reference':external_reference_dose_kg_m3,
                                   'dose_status':'supuesto de protocolo previo; aporte YAN por confirmar'})
    windows = logged_co2_artifact_windows(lab,pump_events,t0)
    full_hourly['co2_valid'] = True
    for mask_start,mask_end in windows:
        # Una mediana horaria se excluye si su intervalo cruza el artefacto.
        intersects = (full_hourly['timestamp'] <= mask_end) & (full_hourly['timestamp']+pd.Timedelta(hours=1) >= mask_start)
        full_hourly.loc[intersects,'co2_valid'] = False
    external_full_observed_parts.append(full_hourly)
    temp = temp.dropna(subset=['T']).sort_values('time_h')
    temperature_grid = np.interp(
        grid, temp["time_h"].to_numpy(dtype=float), temp["T"].to_numpy(dtype=float)
    )
    initials = {
        "X": inherited_x0, "Xd": inherited_xd0,
        "N": float(init["YAN_mg_L"]) * 1e-3,
        "G": float(init["Glucosa_g_L"]), "F": float(init["Fructosa_g_L"]),
        "E": 0.0, "Gly": float(init["Glicerol_g_L"]),
    }
    observations = {state: np.full(len(grid), np.nan) for state in base.STATE_NAMES}
    batch = base.BatchData(
        medium="natural", batch=lab, time=grid, temperature_c=temperature_grid,
        pulses={channel: tuple() for channel in base.INPUT_CHANNELS},
        observations=observations, initials=initials,
    )
    core = base.simulate(batch, theta_full, grid)
    assert core is not None and np.isfinite(core.to_numpy(dtype=float)).all()
    stub = full_hourly[["t_h"]].copy()
    stub["batch"] = lab
    stub["matrix"] = "natural"
    stub["calibratable"] = True
    stub["chemistry_last_h"] = horizon
    caches, _ = co2_cross.build_driver_cache({lab: batch}, {"natural": theta_full}, stub)
    cache = caches[lab]
    pulse_batch = replace(batch,pulses={**batch.pulses,'N':schedule})
    pulse_core = base.simulate(pulse_batch,theta_full,cache.time_h)
    assert pulse_core is not None and np.isfinite(pulse_core.to_numpy(float)).all()
    cache = cache_with_recorded_pulses(cache,pulse_batch,pulse_core,schedule)
    # Comprueba cada salto en N en la hora exacta recuperada del logger.
    probe_times = np.array([0.0,*[t for t,_ in schedule]])
    before = base.simulate(pulse_batch,theta_full,probe_times,sample_event_order='sample_before_action')
    after = base.simulate(pulse_batch,theta_full,probe_times,sample_event_order='action_before_sample')
    for t,amount in schedule:
        assert np.isclose(after.loc[t,'N']-before.loc[t,'N'],amount,rtol=0,atol=1e-12)
    external_batches[lab] = pulse_batch
    external_cache[lab] = cache
    external_initial_rows.append({
        "lab": lab, **initials,
        "X_Xd_source": "media LAB013–LAB015 (mismo mosto)",
        "G_F_N_Gly_source": str(init["Fuente"]),
        "strict_holdout_last_h": strict_holdout_last_h, "record_last_h": horizon,
        'nutrition_events':len(schedule),'first_nutrition_h':schedule[0][0],
        'total_N_kg_m3_reference':sum(a for _,a in schedule),
    })

external_full_observed = pd.concat(external_full_observed_parts, ignore_index=True)
external_initial_table = pd.DataFrame(external_initial_rows)
external_pulse_table = pd.DataFrame(external_pulse_rows)
external_artifact_audit = pd.DataFrame(external_mask_rows)
external_full_valid = external_full_observed[external_full_observed['co2_valid']].copy()
display(external_pulse_table[['lab','start','end','start_h','duration_min','delta_N_kg_m3_reference','dose_status']])
display(external_artifact_audit)
display(Markdown('La nutrición inicial ya está incluida en las condiciones iniciales heredadas. '
                 'La nutrición posterior se aplica en los flancos del log de temperatura. '
                 'Se conserva la dosis de referencia de la prueba anterior (0.14 por evento), '
                 'señalada como supuesto, sin ajustar dosis ni parámetros con CO2 del holdout. '
                 'Los artefactos detectados con CO2 raw se excluyen con la regla previa de recuperación de 90 min.'))
inventory = pd.DataFrame([
    {"grupo": "calibración histórica", "labs": ", ".join(HISTORICAL_TRAIN_LABS), "peso_objetivo_por_lab": AUDIT_CONFIG["historical_weight"], "participa_fit": True},
    {"grupo": "calibración reciente prioritaria", "labs": ", ".join(RECENT_TRAIN_LABS), "peso_objetivo_por_lab": AUDIT_CONFIG["recent_weight"], "participa_fit": True},
    {"grupo": "validación de ajustes completos", "labs": ", ".join(VALIDATION_LABS), "peso_objetivo_por_lab": np.nan, "participa_fit": False},
    {"grupo": "holdout externo", "labs": ", ".join(EXTERNAL_LABS), "peso_objetivo_por_lab": np.nan, "participa_fit": False},
])
display(inventory)
display(external_initial_table)
display(external_support_audit)

evaluation_groups = (
    ("train_historical", HISTORICAL_TRAIN_LABS),
    ("train_recent", RECENT_TRAIN_LABS),
    ("historical_validation", VALIDATION_LABS),
    ("LAB010_sensitivity", SENSITIVITY_LABS),
)


In [ ]:
TRANSIENT_REFERENCE_N = 0.14
transient_frozen_snapshot = {'theta_full': dict(theta_full), 'co2_params': dict(co2_params)}
transient_caches_full, transient_qprod_base, transient_schedules = {}, {}, {}
transient_full_support, transient_support_rows = {}, []
transient_lab_groups = (*evaluation_groups, ('external_holdout', EXTERNAL_LABS))


def full_valid_support(lab):
    if lab in RECENT_TRAIN_LABS:
        group = co2_hourly[co2_hourly['lab'].eq(lab) & co2_hourly['co2_valid']].copy()
        group = group.rename(columns={'co2_observed_g_l_h': 'co2_rate_g_l_h'})
        group['detection_limit_g_l_h'] = co2_cross.CO2_DETECTION_LIMIT_G_L_H
    elif lab in EXTERNAL_LABS:
        # Soporte pre-pulso auditado + tramo posterior limpio, con dosis de referencia explícita.
        pre = external_observed[external_observed['lab'].eq(lab)].copy()
        pre = pre.rename(columns={'time_h':'t_h','co2_observed_g_l_h':'co2_rate_g_l_h'})
        post = external_full_valid[external_full_valid['lab'].eq(lab)
                                  & (external_full_valid['t_h']>=external_nutrition_schedules[lab][0][0])].copy()
        post = post.rename(columns={'co2_observed_g_l_h':'co2_rate_g_l_h'})
        group = pd.concat([pre[['t_h','co2_rate_g_l_h']],post[['t_h','co2_rate_g_l_h']]],ignore_index=True)
        group['detection_limit_g_l_h'] = co2_cross.CO2_DETECTION_LIMIT_G_L_H
        group['left_censored'] = group['co2_rate_g_l_h'] < co2_cross.CO2_DETECTION_LIMIT_G_L_H
    else:
        group = historical_obs[historical_obs['batch'].eq(lab)].copy()
        group = group[group['calibratable']].copy()
    finite = np.isfinite(group['t_h']) & np.isfinite(group['co2_rate_g_l_h'])
    return group[finite & group['t_h'].ge(0)].sort_values('t_h').reset_index(drop=True)


for role, labs in transient_lab_groups:
    for lab in labs:
        if lab in RECENT_TRAIN_LABS:
            cache, batch = recent_cache[lab], holdout_batches[lab]
        elif lab in EXTERNAL_LABS:
            cache, batch = external_cache[lab], external_batches[lab]
        else:
            cache, batch = historical_cache[lab], historical_batches_all[lab]
        core = base.simulate(batch, theta_full, cache.time_h)
        assert core is not None and np.isfinite(core.to_numpy()).all(), lab
        bracket = activity_bracket(core.index, core['G'] + core['F'], core['E'])
        cache = replace(cache, chemical_activity_lower_h=bracket['lower_h'],
                        chemical_activity_upper_h=bracket['upper_h'],
                        chemical_activity_center_h=bracket['center_h'],
                        chemical_activity_scale_h=bracket['scale_h'])
        transient_caches_full[lab] = cache
        _, qprod, _, _ = co2_cross.raw_qgas_grid_prediction(
            cache, k_release_h=co2_params['kCO2_release_h'], sat_scale=co2_params['CO2sat_scale'],
            o2_qmax_mg_gdw_h=co2_params['O2_qmax_mg_gdw_h'],
            o2_initial_scale=co2_params['O2_initial_scale'], chemistry_aligned=True,
            nitrogen_boost_transition=True, pulse_t_rise_h=co2_params['pulse_t_rise_h'],
            pulse_activity_gain=co2_params['pulse_activity_gain'], continuous_release=True,
            bounded_chemical_activation=True,
            chem_activation_start_fraction=co2_params['chem_activation_start_fraction'],
            chem_activation_duration_fraction=co2_params['chem_activation_duration_fraction'])
        transient_qprod_base[lab] = qprod
        transient_schedules[lab] = (
            ((float(cache.n_pulse_time_h), float(cache.n_pulse_amount_kg_m3)),)
            if np.isfinite(cache.n_pulse_time_h) and cache.n_pulse_amount_kg_m3 > 0 else tuple())
        if lab in EXTERNAL_LABS:
            transient_schedules[lab] = external_nutrition_schedules[lab]
        group = full_valid_support(lab)
        assert len(group) >= 3, lab
        assert group['t_h'].max() <= cache.time_h[-1] + 1e-9, lab
        transient_full_support[lab] = group
        pulse_h = cache.n_pulse_time_h
        transient_support_rows.append({
            'lab': lab, 'role': role, 'peso_fit': FIT_LAB_WEIGHT.get(lab, np.nan),
            'n_validos': len(group), 't_final_h': float(group['t_h'].max()),
            'pulso_h': pulse_h, 'dosis_modelada_kg_m3': cache.n_pulse_amount_kg_m3,
            'n_pre_pulso': int((group['t_h'] < pulse_h).sum()) if np.isfinite(pulse_h) else len(group),
            'n_post_pulso': int((group['t_h'] >= pulse_h).sum()) if np.isfinite(pulse_h) else 0,
            'n_cola_24h': int((group['t_h'] >= pulse_h + 24).sum()) if np.isfinite(pulse_h) else 0,
            'gate_lower_predicho_h': bracket['lower_h'], 'gate_upper_predicho_h': bracket['upper_h'],
        })
transient_support_inventory = pd.DataFrame(transient_support_rows)
assert set(TRAIN_LABS).isdisjoint(EXTERNAL_LABS + VALIDATION_LABS)
assert all(transient_support_inventory.set_index('lab').loc[lab, 'n_post_pulso'] > 0 for lab in TRAIN_LABS)
display(transient_support_inventory.round(4))

# Trazabilidad de la dosis, sin convertir masas de producto con un aporte YAN supuesto.
wb_dose = openpyxl.load_workbook(DATA_DIR / 'Mediciones-Offline-LAB019-021.xlsx', read_only=True, data_only=True)
withdrawals = pd.DataFrame(list(wb_dose['MedicionesOffline'].values)[1:],
                           columns=list(next(wb_dose['MedicionesOffline'].values)))
withdrawals['stamp'] = pd.to_datetime(withdrawals['Fecha'].astype(str) + ' ' + withdrawals['Hora'].astype(str), errors='coerce')
withdrawals['ml no retenido'] = pd.to_numeric(withdrawals['ml no retenido'], errors='coerce')
dose_rows = []
for row in pulse2_rows:
    lab = row[0]
    sampled = withdrawals[withdrawals['CodigoLab'].eq(lab) & withdrawals['stamp'].le(PULSE2_DT[lab])]
    dose_rows.append({'lab': lab, 'pulso_h_registrado': PULSE2_H[lab], 'YAN_Excel_mg_L': row[4],
                      'SFX_g': row[5], 'FDA_g': row[6], 'solucion_agregada_mL': row[7],
                      'retiros_registrados_antes_pulso_mL': sampled['ml no retenido'].sum(min_count=1),
                      'muestras_sin_retiro_registrado': int(sampled['ml no retenido'].isna().sum()),
                      'dosis_usada_kg_m3': transient_schedules[lab][0][1]})
wb_dose.close()
display(pd.DataFrame(dose_rows))
display(Markdown('La prueba conserva la concentración de YAN registrada en Nutricion (80 mg/L). '
                 'Los retiros y los 100 mL añadidos se muestran separados: no se supone que el volumen real '
                 'sea 2 L constante ni se reconstruye una concentración a partir de gramos con un factor YAN no confirmado. '
                 'La conversión de flujo conserva la convención histórica de normalización a 2 L para comparar las capas. '
                 'Los balances de volumen y dilución no se recalibran en esta prueba.'))


def normalized_pulse_kernel(time_h, pulse_h, tau_rise, tau_decay):
    u = np.asarray(time_h, dtype=float) - float(pulse_h)
    positive_u = np.maximum(u, 0.0)
    u_peak = tau_rise * np.log1p(tau_decay / tau_rise)
    peak = -np.expm1(-u_peak / tau_rise) * np.exp(-u_peak / tau_decay)
    return np.where(u >= 0, -np.expm1(-positive_u / tau_rise) * np.exp(-positive_u / tau_decay) / peak, 0.0)


## Ecuaciones de las seis variantes

$q_0,X_0$: simulaciones sin nutrición posterior; $q_N,X_N$: con las adiciones registradas. Nutrición inicial incluida en ICs. Para la rampa:
$$r(t)=\begin{cases}0&t<t_p\\(t-t_p)/\tau_p&t_p\le t<t_p+\tau_p\\1&t\ge t_p+\tau_p\end{cases}$$

| Variante | Producción biológica $q_b$ | Biomasa del bloque O2 |
|---|---|---|
| Actual | $\max\{[1+(g_p-1)r]q_0+r(q_N-q_0),0\}$ | $\max\{X_0+r(X_N-X_0),0\}$ |
| Rampa sin atenuación | $\max\{q_0+r(q_N-q_0),0\}$ | $\max\{X_0+r(X_N-X_0),0\}$ |
| Directa | $\max\{q_N,0\}$ | $\max\{X_N,0\}$ |

Con O2:
$$v_O=q_{O,max}X\frac{O}{K_O+O},\quad O_{j+1}=\max(O_j-\Delta t_jv_{O,j},0),\quad O_0=s_{O,0}O_{sat}(0).$$
$$f_{ferm}=f_{min}+(1-f_{min})\frac{K_{ana}^{n}}{K_{ana}^{n}+O^{n}},\quad q_{resp}=Y_{CO2/O2}v_O.$$
Sin O2: $f_{ferm}=1$, $q_{resp}=0$.

Activación: $t_a=t_L+\alpha(t_U-t_L)$, $d_a=\max[\delta(t_U-t_L),0.25]$:
$$a(t)=\begin{cases}0&t<t_a\\3[(t-t_a)/d_a]^2-2[(t-t_a)/d_a]^3&t_a\le t<t_a+d_a\\1&t\ge t_a+d_a\end{cases}$$

Transitorio: $u=t-t_{p,i}$, $u_*=\tau_r\ln(1+\tau_d/\tau_r)$, $K=(1-e^{-u_*/\tau_r})e^{-u_*/\tau_d}$:
$$h(u)=\begin{cases}0&u<0\\(1-e^{-u/\tau_r})e^{-u/\tau_d}/K&u\ge0\end{cases}$$
$$q_{in}=a(t)[f_{ferm}q_b+q_{resp}]\left[1+A\sum_i\frac{\Delta N_i}{0.14}h(t-t_{p,i})\right].$$
$$q_{out}=kC\left[0.05+0.95\frac{C}{C+C_*}\right],\quad \frac{dC}{dt}=q_{in}-q_{out},\quad q_{pred}=g_{sensor}q_{out}.$$

Salida limitada al inventario disponible en cada paso. La ganancia del sensor no entra al balance del pool. El transitorio es empírico: no agrega consumo de azúcar a la cinética. O2, fracción fermentativa y respiración son trayectorias calculadas, no mediciones.

In [ ]:
VARIANTS={}
for mode,label,color in (('actual','Actual','#0072B2'),('rampa_sin_atenuacion','Rampa g=1','#D55E00'),('directa','Directa','#009E73')):
    for oxygen in (True,False):
        key=mode+('_con_O2' if oxygen else '_sin_O2')
        VARIANTS[key]={'nutrition_mode':mode,'oxygen_enabled':oxygen,'label':label+(' con O2' if oxygen else ' sin O2'),
                       'color':color,'linestyle':'-' if oxygen else '--'}
ALL_LABS=tuple(transient_caches_full)
BASELINE_VARIANT='actual_con_O2'
POOL_FIT_KEYS=('kCO2_release_h','CO2sat_scale','matrix_gain','A','tau_rise_h','tau_decay_h')


def production_layers(lab,nutrition_mode,oxygen_enabled,overrides=None):
    p={**co2_params,'activation_enabled':True,'activation_shift_h':0.0,**(overrides or {})}
    assert nutrition_mode in ('actual','rampa_sin_atenuacion','directa')
    assert p['pulse_t_rise_h']>0 and p['pulse_activity_gain']>=0
    assert p['O2_qmax_mg_gdw_h']>0 and p['O2_initial_scale']>=0
    cache=transient_caches_full[lab];t=cache.time_h
    elapsed=t-cache.n_pulse_time_h
    r=np.where(elapsed<0,0.0,np.where(elapsed>=p['pulse_t_rise_h'],1.0,elapsed/p['pulse_t_rise_h']))
    if not np.isfinite(cache.n_pulse_time_h) or cache.n_pulse_amount_kg_m3<=0:r=np.zeros(len(t))
    q0=cache.base_qprod_no_n_pulse_g_l_h
    if nutrition_mode=='directa':
        qb=np.maximum(cache.base_qprod_g_l_h,0.0);biomass=np.maximum(cache.biomass_g_l,0.0);activity=np.ones(len(t))
    else:
        activity=1.0+(p['pulse_activity_gain']-1)*r if nutrition_mode=='actual' else np.ones(len(t))
        qb=np.maximum(activity*q0+r*cache.n_pulse_qprod_increment_g_l_h,0.0)
        biomass=np.maximum(cache.biomass_no_n_pulse_g_l+r*cache.n_pulse_biomass_increment_g_l,0.0)
    uptake=np.zeros(len(t))
    if oxygen_enabled:
        oxygen=np.zeros(len(t));oxygen[0]=p['O2_initial_scale']*cache.o2_saturation_base_mg_l[0]
        for j in range(len(t)):
            uptake[j]=p['O2_qmax_mg_gdw_h']*biomass[j]*oxygen[j]/(co2_cross.O2_K_MG_L+oxygen[j])
            if j<len(t)-1:oxygen[j+1]=max(oxygen[j]-(t[j+1]-t[j])*uptake[j],0.0)
        phi=co2_cross.O2_ANA_K_MG_L**co2_cross.O2_ANA_HILL/(
            co2_cross.O2_ANA_K_MG_L**co2_cross.O2_ANA_HILL+np.maximum(oxygen,0)**co2_cross.O2_ANA_HILL)
        fferm=co2_cross.O2_CRABTREE_FLOOR+(1-co2_cross.O2_CRABTREE_FLOOR)*phi
        qresp=co2_cross.co2_model.CO2_G_PER_MG_O2_RESP*uptake
    else:
        oxygen=np.full(len(t),np.nan);fferm=np.ones(len(t));qresp=np.zeros(len(t))
    lower,upper=cache.chemical_activity_lower_h,cache.chemical_activity_upper_h
    if not p['activation_enabled']:activation=np.ones(len(t))
    elif upper<=lower+1e-9:activation=(t>=upper+p['activation_shift_h']).astype(float)
    else:
        start=lower+p['chem_activation_start_fraction']*(upper-lower)+p['activation_shift_h']
        duration=max(p['chem_activation_duration_fraction']*(upper-lower),co2_cross.DRIVER_GRID_H)
        f=(t-start)/duration
        activation=np.where(t<start,0.0,np.where(t>=start+duration,1.0,3*f**2-2*f**3))
    qbase=np.maximum(activation*(qb*fferm+qresp),0.0)
    return {'t_h':t,'q0':q0,'qbio':qb,'ramp':r,'activity':activity,'O2_mg_l':oxygen,
            'uptake':uptake,'f_ferm':fferm,'qresp':qresp,'activation':activation,'qbase':qbase}


def pool_prediction(lab,layers,params,return_curve=True):
    p={**FROZEN_POOL_PARAMETERS,**params}
    for name in ('kCO2_release_h','CO2sat_scale','matrix_gain','tau_rise_h','tau_decay_h'):
        assert np.isfinite(p[name]) and p[name]>0,name
    assert np.isfinite(p['A']) and p['A']>=0
    cache=transient_caches_full[lab];t=cache.time_h
    bump=np.zeros(len(t))
    for event,dose in transient_schedules[lab]:
        bump+=dose/TRANSIENT_REFERENCE_N*normalized_pulse_kernel(t,event,p['tau_rise_h'],p['tau_decay_h'])
    multiplier=1+p['A']*bump;qin=layers['qbase']*multiplier;csat=p['CO2sat_scale']*cache.csat_base_g_l
    tk=cache.temperature_used_c+273.15;tref=p['release_reference_temperature_C']+273.15
    assert tref>0 and (tk>0).all()
    k=p['kCO2_release_h']*np.exp(p['release_temperature_sensitivity_K']*(1/tref-1/tk))
    assert np.isfinite(k).all() and (k>0).all(),'sensibilidad térmica fuera de rango numérico'
    c=np.zeros(len(t));qout=np.zeros(len(t))
    for j,dt in enumerate(np.diff(t),start=1):
        previous=c[j-1]
        frac=co2_cross.CO2_CONTINUOUS_RELEASE_FLOOR+(1-co2_cross.CO2_CONTINUOUS_RELEASE_FLOOR)*previous/max(previous+csat[j],1e-12)
        qout[j]=min(k[j]*previous*frac,previous/dt+qin[j-1])
        c[j]=max(previous+dt*(qin[j-1]-qout[j]),0.0)
    predicted=p['matrix_gain']*qout
    if not return_curve:return predicted
    mass_error=np.max(np.abs(np.diff(c)-np.diff(t)*(qin[:-1]-qout[1:])))
    assert mass_error<1e-10 and (c>=0).all() and (qout>=0).all()
    return pd.DataFrame({**layers,'q_input':qin,'transient_multiplier':multiplier,
                         'dissolved_g_l':c,'q_output':qout,'predicted':predicted,'k_release_h':k})


formal_layers={(v,lab):production_layers(lab,s['nutrition_mode'],s['oxygen_enabled']) for v,s in VARIANTS.items() for lab in ALL_LABS}
# Regression check of the official reference and conservation of recorded N pulses.
for lab in ALL_LABS:
    cache=transient_caches_full[lab]
    expected,expected_o2,_=co2_cross.effective_qprod_grid(
        cache,co2_params['O2_qmax_mg_gdw_h'],co2_params['O2_initial_scale'],True,
        nitrogen_boost_transition=True,pulse_t_rise_h=co2_params['pulse_t_rise_h'],pulse_activity_gain=co2_params['pulse_activity_gain'],
        bounded_chemical_activation=True,chem_activation_start_fraction=co2_params['chem_activation_start_fraction'],
        chem_activation_duration_fraction=co2_params['chem_activation_duration_fraction'])
    assert np.allclose(formal_layers[(BASELINE_VARIANT,lab)]['qbase'],expected,rtol=1e-11,atol=1e-12),lab
    assert np.allclose(formal_layers[(BASELINE_VARIANT,lab)]['O2_mg_l'],expected_o2,rtol=1e-11,atol=1e-12),lab
    official,*_=co2_cross.raw_qgas_grid_prediction(
        cache,k_release_h=co2_params['kCO2_release_h'],sat_scale=co2_params['CO2sat_scale'],
        o2_qmax_mg_gdw_h=co2_params['O2_qmax_mg_gdw_h'],o2_initial_scale=co2_params['O2_initial_scale'],
        chemistry_aligned=True,nitrogen_boost_transition=True,pulse_t_rise_h=co2_params['pulse_t_rise_h'],
        pulse_activity_gain=co2_params['pulse_activity_gain'],continuous_release=True,bounded_chemical_activation=True,
        chem_activation_start_fraction=co2_params['chem_activation_start_fraction'],
        chem_activation_duration_fraction=co2_params['chem_activation_duration_fraction'])
    rebuilt=pool_prediction(lab,formal_layers[(BASELINE_VARIANT,lab)],FROZEN_POOL_PARAMETERS)
    assert np.allclose(rebuilt['predicted'],co2_params['matrix_gain']*official,rtol=1e-10,atol=1e-12),lab
    direct=formal_layers[('directa_sin_O2',lab)]
    assert np.allclose(direct['qbase'],direct['activation']*np.maximum(cache.base_qprod_g_l_h,0))
print('Referencia histórica reproducida para todos los LAB; variante directa conserva pulsos cinéticos.')
display(pd.DataFrame([{'variante':key,**v} for key,v in VARIANTS.items()]).drop(columns=['color','linestyle']))


## Etapa 1: estructuras con parámetros congelados

Mismos parámetros archivados de liberación/ganancia y A=0. Ningún ajuste compensa el cambio de bloques.

In [ ]:
role_by_lab={lab:role for role,labs in transient_lab_groups for lab in labs}


def evaluate_curves(curves,stage):
    rows=[]
    for (variant,lab),curve in curves.items():
        group=transient_full_support[lab];events=transient_schedules[lab]
        windows={'completo':np.ones(len(group),bool)}
        if events:
            times=group['t_h'].to_numpy(float);pulse=np.zeros(len(group),bool)
            for event,_ in events:pulse|=(times>=event)&(times<=event+24)
            windows.update(pre_pulso=times<events[0][0],pulso_24h=pulse,cola=times>events[-1][0]+24)
        for window,mask in windows.items():
            part=group[mask]
            if len(part)<3:continue
            pred=np.interp(part['t_h'],curve['t_h'],curve['predicted'])
            row=metric_row(variant,lab,part,pred)
            if window not in ('completo','pre_pulso'):
                for field in ('onset_threshold_g_l_h','onset_observed_h','onset_predicted_h','onset_error_h'):row[field]=np.nan
            row.update(stage=stage,role=role_by_lab[lab],window=window);rows.append(row)
    return pd.DataFrame(rows)


def plot_audit_curves(curves,labs,title,ncols=3,variants=None):
    chosen=tuple(VARIANTS) if variants is None else tuple(variants);nr=math.ceil(len(labs)/ncols)
    fig=plt.figure(figsize=(16 if ncols==3 else 13,4*nr+1.3),layout='constrained')
    gs=fig.add_gridspec(nr+1,ncols,height_ratios=[0.27]+[1]*nr)
    header=fig.add_subplot(gs[0,:]);header.set_axis_off();axes=[]
    for i,lab in enumerate(labs):
        ax=fig.add_subplot(gs[1+i//ncols,i%ncols]);obs=transient_full_support[lab]
        ax.plot(obs['t_h'],obs['co2_rate_g_l_h'],color='black',lw=1.65,label='Observado válido')
        for variant in chosen:
            v=VARIANTS[variant];c=curves[(variant,lab)]
            ax.plot(c['t_h'],c['predicted'],color=v['color'],ls=v['linestyle'],lw=1.8,label=v['label'])
        for event,_ in transient_schedules[lab]:ax.axvline(event,color='0.75',ls=':',lw=0.8)
        ax.set_title(lab+(f' (peso {FIT_LAB_WEIGHT[lab]:g})' if lab in FIT_LAB_WEIGHT else ''))
        ax.set_xlim(0,float(c['t_h'].max()));ax.set_ylim(bottom=0);ax.set_xlabel('t desde inoculación [h]')
        if i%ncols==0:ax.set_ylabel(r'Flujo CO$_2$ [g L$^{-1}$ h$^{-1}$]')
        ax.grid(alpha=0.25);axes.append(ax)
    h,l=axes[0].get_legend_handles_labels();header.legend(h,l,loc='center',ncol=4,frameon=False,fontsize=9)
    fig.suptitle(title,fontsize=14);display(fig);plt.close(fig)


frozen_curves={(v,lab):pool_prediction(lab,formal_layers[(v,lab)],FROZEN_POOL_PARAMETERS) for v in VARIANTS for lab in ALL_LABS}
frozen_metrics=evaluate_curves(frozen_curves,'congelado')
display(frozen_metrics[frozen_metrics['window'].eq('completo')].groupby(['role','model'],as_index=False).agg(
    NRMSE_mediano=('NRMSE_peak','median'),RMSE_mediano=('RMSE_g_l_h','median'),
    error_onset_abs_mediano_h=('onset_error_h',lambda x:float(np.nanmedian(np.abs(x))))).round(4))
plot_audit_curves(frozen_curves,TRAIN_LABS,'Estructuras con parámetros congelados y sin transitorio')


## Etapa 2: reajuste comparable

Se ajustan $k$, escala de solubilidad, ganancia, $A$, $\tau_r$, $\tau_d$. Mismos límites, inicios y curvas válidas completas para las seis variantes; residuos con censura izquierda, normalización por pico y número de muestras y peso por LAB. Bloques O2/rampa/activación permanecen congelados. Selección únicamente por NRMSE mediano completo de LAB011–LAB012; el holdout se evalúa después.

In [ ]:
audit_fit_blocks={}
for lab in TRAIN_LABS:
    group=transient_full_support[lab];obs=group['co2_rate_g_l_h'].to_numpy(float)
    scale=max(float(obs.max()),co2_cross.CO2_RESIDUAL_SIGMA_FLOOR_G_L_H)
    audit_fit_blocks[lab]=(group['t_h'].to_numpy(float),obs,group['left_censored'].astype(bool).to_numpy(),
                          group['detection_limit_g_l_h'].to_numpy(float),math.sqrt(FIT_LAB_WEIGHT[lab]/len(group))/scale)
assert set(audit_fit_blocks)==set(TRAIN_LABS)
assert set(audit_fit_blocks).isdisjoint(VALIDATION_LABS+EXTERNAL_LABS+SENSITIVITY_LABS)


def decode_fit(vector):
    p=dict(FROZEN_POOL_PARAMETERS)
    for i,key in enumerate(POOL_FIT_KEYS):p[key]=float(vector[i]) if key=='A' else float(np.exp(vector[i]))
    return p


def audit_residual(vector,variant):
    params=decode_fit(vector);blocks=[]
    for lab,(time,obs,censored,limit,weight) in audit_fit_blocks.items():
        pred_grid=pool_prediction(lab,formal_layers[(variant,lab)],params,return_curve=False)
        pred=np.interp(time,transient_caches_full[lab].time_h,pred_grid)
        blocks.append(weight*np.where(censored,np.maximum(pred-limit,0),pred-obs))
    return np.concatenate(blocks)


fit_results={};fit_start_rows=[];parameter_rows=[]
if AUDIT_CONFIG['run_refits']:
    lower=[];upper=[]
    for key in POOL_FIT_KEYS:
        lo,hi=AUDIT_CONFIG['bounds'][key];assert lo<hi and (key=='A' or lo>0)
        lower.append(lo if key=='A' else np.log(lo));upper.append(hi if key=='A' else np.log(hi))
    for variant in VARIANTS:
        best=None
        for start_index,(amplitude,rise,decay) in enumerate(AUDIT_CONFIG['multistart']):
            seed={**FROZEN_POOL_PARAMETERS,'A':amplitude,'tau_rise_h':rise,'tau_decay_h':decay}
            x=[seed[k] if k=='A' else np.log(seed[k]) for k in POOL_FIT_KEYS]
            x=np.minimum(np.maximum(x,np.array(lower)+1e-10),np.array(upper)-1e-10)
            result=least_squares(audit_residual,x,args=(variant,),bounds=(lower,upper),
                                 max_nfev=AUDIT_CONFIG['max_nfev'],ftol=1e-7,xtol=1e-7,gtol=1e-7)
            wsse=float(result.fun@result.fun);params=decode_fit(result.x)
            fit_start_rows.append({'variant':variant,'start':start_index,'success':result.success,'nfev':result.nfev,
                                   'WSSE':wsse,**{k:params[k] for k in POOL_FIT_KEYS}})
            print(variant,'inicio',start_index,'WSSE',round(wsse,6),'convergencia',result.success,flush=True)
            if result.success and (best is None or wsse<best['WSSE']):best={'params':params,'WSSE':wsse}
        assert best is not None,f'{variant}: ningún inicio convergió'
        fit_results[variant]=best
        for key in POOL_FIT_KEYS:
            value=best['params'][key];lo,hi=AUDIT_CONFIG['bounds'][key]
            d=min(value-lo,hi-value)/(hi-lo) if key=='A' else min(np.log(value/lo),np.log(hi/value))/np.log(hi/lo)
            parameter_rows.append({'variant':variant,'parameter':key,'estimate':value,'lower':lo,'upper':hi,'active_bound':d<=0.01})
else:
    fit_results={v:{'params':dict(FROZEN_POOL_PARAMETERS),'WSSE':np.nan} for v in VARIANTS}
fit_start_table=pd.DataFrame(fit_start_rows);fit_parameter_table=pd.DataFrame(parameter_rows)
display(fit_parameter_table.round(5));display(fit_start_table.round(5))


## Validación y selección independiente


In [ ]:
refit_stage='reajustado' if AUDIT_CONFIG['run_refits'] else 'congelado_sin_ajuste'
refit_curves={(v,lab):pool_prediction(lab,formal_layers[(v,lab)],fit_results[v]['params']) for v in VARIANTS for lab in ALL_LABS}
refit_metrics=evaluate_curves(refit_curves,refit_stage)
full_metrics=refit_metrics[refit_metrics['window'].eq('completo')]
validation_ranking=(full_metrics[full_metrics['role'].eq('historical_validation')].groupby('model',as_index=False).agg(
    median_NRMSE=('NRMSE_peak','median'),median_abs_onset_h=('onset_error_h',lambda x:float(np.nanmedian(np.abs(x)))),
    median_abs_AUC=('AUC_relative_error',lambda x:float(np.nanmedian(np.abs(x))))))
validation_ranking=validation_ranking.sort_values('median_NRMSE').reset_index(drop=True)
SELECTED_VARIANT=str(validation_ranking.iloc[0]['model']);assert set(validation_ranking['model'])==set(VARIANTS)
display(validation_ranking.round(4));print('Selección usando únicamente LAB011–LAB012:',SELECTED_VARIANT)
display(full_metrics.groupby(['role','model'],as_index=False).agg(
    median_RMSE=('RMSE_g_l_h','median'),median_NRMSE=('NRMSE_peak','median'),
    median_abs_onset_h=('onset_error_h',lambda x:float(np.nanmedian(np.abs(x))))).round(4))
display(refit_metrics[refit_metrics['role'].str.startswith('train')&refit_metrics['window'].ne('completo')]
        .groupby(['model','window'],as_index=False).agg(median_RMSE=('RMSE_g_l_h','median'),
                                                      median_abs_AUC=('AUC_relative_error',lambda x:float(np.nanmedian(np.abs(x))))).round(4))
plot_audit_curves(refit_curves,TRAIN_LABS,'Reajuste comparable: curvas completas de calibración')
plot_audit_curves(refit_curves,VALIDATION_LABS,'Validación independiente LAB011–LAB012',ncols=2)


## Holdout externo LAB016–LAB018

La selección anterior ya está fijada. Se muestran todas las alternativas sin modificarla a partir del holdout.


In [ ]:
external_audit_metrics=refit_metrics[refit_metrics['role'].eq('external_holdout')]
display(external_audit_metrics[['model','lab','window','n','RMSE_g_l_h','NRMSE_peak','onset_error_h','AUC_relative_error']].round(4))
plot_audit_curves(refit_curves,EXTERNAL_LABS,'Holdout externo con nutrición registrada (dosis de referencia)',ncols=1)


## Activación inicial y contribución del oxígeno

Se retira a(t) por separado para la referencia y la variante seleccionada, sin reajuste. Esta prueba no participa en la selección. La contribución O2 usa integrales de entrada al pool antes de ganancia y transitorio; no valida su trayectoria fisiológica.

In [ ]:
gate_variants=tuple(dict.fromkeys((BASELINE_VARIANT,SELECTED_VARIANT)));gate_curves={};gate_rows=[]
for variant in gate_variants:
    v=VARIANTS[variant]
    for lab in (*RECENT_TRAIN_LABS,*VALIDATION_LABS,*EXTERNAL_LABS):
        layers=production_layers(lab,v['nutrition_mode'],v['oxygen_enabled'],{'activation_enabled':False})
        off=pool_prediction(lab,layers,fit_results[variant]['params']);on=refit_curves[(variant,lab)]
        gate_curves[(variant,lab)]=off;group=transient_full_support[lab]
        for label,curve in (('con_a',on),('sin_a',off)):
            row=metric_row(variant,lab,group,np.interp(group['t_h'],curve['t_h'],curve['predicted']))
            row.update(gate=label,role=role_by_lab[lab]);gate_rows.append(row)
gate_metrics=pd.DataFrame(gate_rows)
display(gate_metrics[['model','lab','gate','RMSE_g_l_h','onset_error_h','AUC_relative_error']].round(4))
fig=plt.figure(figsize=(16,9.2),layout='constrained');gs=fig.add_gridspec(3,3,height_ratios=[0.22,1,1])
header=fig.add_subplot(gs[0,:]);header.set_axis_off()
for i,lab in enumerate((*RECENT_TRAIN_LABS,*VALIDATION_LABS)):
    ax=fig.add_subplot(gs[1+i//3,i%3]);obs=transient_full_support[lab]
    ax.plot(obs['t_h'],obs['co2_rate_g_l_h'],color='black',lw=1.6,label='Observado válido')
    for gate_index,variant in enumerate(gate_variants):
        v=VARIANTS[variant]
        gate_color=('#0072B2','#D55E00')[gate_index]
        for label,style,curve in (('con a(t)','-',refit_curves[(variant,lab)]),('sin a(t)',':',gate_curves[(variant,lab)])):
            ax.plot(curve['t_h'],curve['predicted'],color=gate_color,ls=style,lw=1.8,label=v['label']+' '+label)
    ax.set_title(lab);ax.set_xlim(0,curve['t_h'].max());ax.set_ylim(bottom=0);ax.set_xlabel('t [h]')
    if i%3==0:ax.set_ylabel(r'Flujo CO$_2$ [g L$^{-1}$ h$^{-1}$]')
    if i==0:
        h,l=ax.get_legend_handles_labels();header.legend(h,l,ncol=3,loc='center',frameon=False,fontsize=8.5)
fig.suptitle('Activación inicial: retirar a(t), sin reajuste',fontsize=14);display(fig);plt.close(fig)
oxygen_rows=[]
for lab in ALL_LABS:
    on=formal_layers[('actual_con_O2',lab)];off=formal_layers[('actual_sin_O2',lab)];dt=np.diff(on['t_h'])
    row={'lab':lab,'role':role_by_lab[lab],'O2_inicial_calculado_mg_L':on['O2_mg_l'][0],
         'horas_f_ferm_menor_0_95':float(np.sum(dt*(on['f_ferm'][:-1]<0.95))),
         'CO2_respiratorio_activado_integrado_g_L':float(np.sum(dt*(on['activation']*on['qresp'])[:-1])),
         'entrada_con_O2_integrada_g_L':float(np.sum(dt*on['qbase'][:-1])),
         'entrada_sin_O2_integrada_g_L':float(np.sum(dt*off['qbase'][:-1]))}
    row['diferencia_entrada_O2_g_L']=row['entrada_con_O2_integrada_g_L']-row['entrada_sin_O2_integrada_g_L'];oxygen_rows.append(row)
oxygen_contribution=pd.DataFrame(oxygen_rows);display(oxygen_contribution.round(4))
fig=plt.figure(figsize=(16,8.7),layout='constrained');gs=fig.add_gridspec(3,3,height_ratios=[0.2,1,1])
header=fig.add_subplot(gs[0,:]);header.set_axis_off()
for i,lab in enumerate(RECENT_TRAIN_LABS):
    p=formal_layers[(BASELINE_VARIANT,lab)]
    ax=fig.add_subplot(gs[1,i]);ax.plot(p['t_h'],p['O2_mg_l'],color='#0072B2')
    ax.set_title(lab);ax.set_xlim(0,p['t_h'].max());ax.set_ylim(bottom=0);ax.set_xlabel('t [h]')
    if i==0:ax.set_ylabel('O2 calculado [mg/L]')
    ax=fig.add_subplot(gs[2,i]);ax.plot(p['t_h'],p['f_ferm'],color='#D55E00',label='Fracción fermentativa')
    ax.plot(p['t_h'],p['activation'],color='0.5',ls='--',label='Activación a(t)')
    ax.set_xlim(0,p['t_h'].max());ax.set_ylim(-0.02,1.05);ax.set_xlabel('t [h]')
    if i==0:
        ax.set_ylabel('Factores adimensionales');h,l=ax.get_legend_handles_labels();header.legend(h,l,ncol=2,loc='center',frameon=False)
fig.suptitle('Bloque de oxígeno y activación: trayectorias calculadas completas',fontsize=14);display(fig);plt.close(fig)
# Temperature windows set by recorded SP changes, not prediction errors.
thermal_rows=[]
for lab in RECENT_TRAIN_LABS:
    f=FERMENTER[lab]
    frame=temperature_raw[temperature_raw['fermentador'].eq(f)].sort_values('timestamp').set_index('timestamp')[['T','SP']]
    frame=frame.resample('1min').median().dropna(subset=['SP']);frame['t_h']=(frame.index-T0).total_seconds()/3600
    changes=frame[frame['SP'].shift().notna()&frame['SP'].ne(frame['SP'].shift())&frame['t_h'].ge(0)]
    obs=transient_full_support[lab]
    for event in changes.itertuples():
        t=float(event.t_h)
        before=obs[obs['t_h'].between(t-3,t-1)]['co2_rate_g_l_h'].median()
        after=obs[obs['t_h'].between(t+3,t+5)]['co2_rate_g_l_h'].median()
        for variant in VARIANTS:
            curve=refit_curves[(variant,lab)]
            pred_before=curve[curve['t_h'].between(t-3,t-1)]['predicted'].median()
            pred_after=curve[curve['t_h'].between(t+3,t+5)]['predicted'].median()
            thermal_rows.append({'lab':lab,'SP_event_h':t,'SP_new_C':event.SP,'variant':variant,
                                 'observed_before':before,'observed_after':after,'predicted_before':pred_before,
                                 'predicted_after':pred_after,'ratio_observed':after/before if before>0 else np.nan,
                                 'ratio_predicted':pred_after/pred_before if pred_before>0 else np.nan})
thermal_audit=pd.DataFrame(thermal_rows);display(thermal_audit.round(4))
matched_rows=[]
for mode in ('actual','rampa_sin_atenuacion','directa'):
    for role in ('train_historical','train_recent','historical_validation','external_holdout'):
        part=full_metrics[full_metrics['role'].eq(role)]
        on=part[part['model'].eq(mode+'_con_O2')]['NRMSE_peak'].median();off=part[part['model'].eq(mode+'_sin_O2')]['NRMSE_peak'].median()
        matched_rows.append({'nutricion':mode,'grupo':role,'NRMSE_con_O2':on,'NRMSE_sin_O2':off,'diferencia_sin_menos_con':off-on})
matched_oxygen_comparison=pd.DataFrame(matched_rows);display(matched_oxygen_comparison.round(4))
if len(fit_parameter_table):display(fit_parameter_table[fit_parameter_table['active_bound']].round(5))
selected=full_metrics[full_metrics['model'].eq(SELECTED_VARIANT)]
baseline=full_metrics[full_metrics['model'].eq(BASELINE_VARIANT)]
external_best=selected[selected['role'].eq('external_holdout')]['NRMSE_peak'].median()
external_baseline=baseline[baseline['role'].eq('external_holdout')]['NRMSE_peak'].median()
display(Markdown(
    f'**Selección por validación: {VARIANTS[SELECTED_VARIANT]["label"]}.** '
    f'NRMSE mediano validación: {validation_ranking.iloc[0]["median_NRMSE"]:.4f}. '
    f'NRMSE mediano externo: {external_best:.4f}; referencia actual reajustada: {external_baseline:.4f}. '
    'El ranking mide predicción de flujo, no demuestra que O2 latente sea correcto. '
    'Leer juntos límites activos, onset, pulso, cola y cambios térmicos. Eafe no se recalibra. '
    'Las ventanas térmicas comparan 3–1 h antes y 3–5 h después: incluyen evolución de la fermentación. '
    'El holdout conserva una dosis de referencia pendiente de confirmar.'))


## Explorador manual — ejecutar después de cambiar `MANUAL`

Recalcula los bloques y el pool sin optimizar ni alterar el ranking. Arriba: flujo observado, comparador formal de la misma variante y curva manual. Centro: producción biológica, entrada y salida del pool antes de ganancia. Abajo: rampa, actividad, activación, fracción fermentativa y transitorio. Figuras completas; el transitorio puede superar 1.

In [ ]:
manual_signature_before=(validation_ranking.to_json(),fit_parameter_table.to_json())
manual_variant=MANUAL['nutrition_mode']+('_con_O2' if MANUAL['oxygen_enabled'] else '_sin_O2')
assert manual_variant in VARIANTS
manual_labs=tuple(MANUAL['labs']);assert manual_labs and set(manual_labs).issubset(ALL_LABS)
manual_curves={};manual_metric_rows=[]
for lab in manual_labs:
    layers=production_layers(lab,MANUAL['nutrition_mode'],MANUAL['oxygen_enabled'],MANUAL)
    curve=pool_prediction(lab,layers,MANUAL);manual_curves[lab]=curve
    group=transient_full_support[lab]
    row=metric_row('manual',lab,group,np.interp(group['t_h'],curve['t_h'],curve['predicted']))
    row['comparador_formal']=manual_variant;manual_metric_rows.append(row)
display(pd.DataFrame([{'control':key,'valor':str(value)} for key,value in MANUAL.items()]))
display(pd.DataFrame(manual_metric_rows)[['lab','RMSE_g_l_h','NRMSE_peak','onset_error_h','AUC_relative_error']].round(4))
for start in range(0,len(manual_labs),3):
    shown=manual_labs[start:start+3];nc=len(shown)
    fig=plt.figure(figsize=(5.3*nc,12.5),layout='constrained');gs=fig.add_gridspec(4,nc,height_ratios=[0.26,1,1,0.85])
    header=fig.add_subplot(gs[0,:]);header.set_axis_off()
    for i,lab in enumerate(shown):
        c=manual_curves[lab];ref=refit_curves[(manual_variant,lab)];obs=transient_full_support[lab]
        ax=fig.add_subplot(gs[1,i]);ax.plot(obs['t_h'],obs['co2_rate_g_l_h'],color='black',lw=1.6,label='Observado válido')
        ax.plot(ref['t_h'],ref['predicted'],color='0.5',ls='--',lw=1.5,label='Referencia formal')
        ax.plot(c['t_h'],c['predicted'],color='#CC79A7',lw=2,label='Prueba manual')
        for event,_ in transient_schedules[lab]:ax.axvline(event,color='0.7',ls=':',lw=0.8)
        ax.set_title(lab);ax.set_xlim(0,c['t_h'].max());ax.set_ylim(bottom=0);ax.set_xlabel('t [h]')
        if i==0:
            ax.set_ylabel(r'Flujo CO$_2$ [g L$^{-1}$ h$^{-1}$]');h,l=ax.get_legend_handles_labels()
            header.legend(h,l,ncol=3,loc='center',frameon=False)
        ax=fig.add_subplot(gs[2,i]);ax.plot(c['t_h'],c['qbio'],color='#0072B2',label='Producción biológica')
        ax.plot(c['t_h'],c['q_input'],color='#D55E00',label='Entrada al pool')
        ax.plot(c['t_h'],c['q_output'],color='#009E73',label='Salida del pool')
        ax.set_xlim(0,c['t_h'].max());ax.set_ylim(bottom=0);ax.set_xlabel('t [h]')
        if i==0:ax.set_ylabel(r'CO$_2$ [g L$^{-1}$ h$^{-1}$]')
        ax.legend(loc='upper right',fontsize=7.5)
        ax=fig.add_subplot(gs[3,i])
        for field,label,color,style in (('ramp','r(t)','#0072B2','-'),('activity','Actividad','#D55E00','-'),
            ('activation','a(t)','0.5','--'),('f_ferm','Fracción fermentativa','#009E73','-'),('transient_multiplier','Transitorio','#CC79A7','-')):
            ax.plot(c['t_h'],c[field],color=color,ls=style,lw=1.4,label=label)
        ax.set_xlim(0,c['t_h'].max());ax.set_ylim(bottom=0);ax.set_xlabel('t desde inoculación [h]')
        if i==0:ax.set_ylabel('Factores adimensionales')
        ax.legend(loc='upper right',fontsize=7)
    fig.suptitle('Exploración manual: '+VARIANTS[manual_variant]['label'],fontsize=14);display(fig);plt.close(fig)
assert manual_signature_before==(validation_ranking.to_json(),fit_parameter_table.to_json())
print('Exploración manual independiente: ranking y parámetros formales intactos.')


In [ ]:
protected_after={str(path):sha256(path) for path in PROTECTED_PATHS}
assert protected_before==protected_after,'Se modificó un artefacto protegido'
assert frozen_theta_snapshot==theta_full and frozen_co2_snapshot==co2_params
assert set(audit_fit_blocks).isdisjoint(VALIDATION_LABS+EXTERNAL_LABS)
print('theta_full, parámetros archivados y notebooks anteriores conservan sus valores/SHA-256.')
print('Imágenes/CSV escritos en results: 0. Ajustes y exploración sólo en memoria y outputs.')
